# WebNLG verbalisation evaluation

This notebook reads generation CSVs and evaluates each generated verbalisation against **all references** attached to the entry.

Metrics computed per instance:
- **ROUGE-L F1**
- **METEOR**
- **chrF++**
- **BERTScore F1**
- **BERT cosine** using `bert-base-multilingual-cased`
- **Cosine similarity** using `intfloat/multilingual-e5-base`
- **Expansion ratio**, selecting the reference whose ratio is **closest to 1.0**

Aggregation:
- summary by **model × language**
- optional summary by **model × language × split**

Matching policy:
- for lexical-overlap and semantic similarity metrics, the notebook compares the candidate with **all references** for that entry and keeps the **best score**.
- for expansion ratio, it keeps the ratio from the reference with `abs(ratio - 1)` minimal.


In [1]:
# Optional: install missing packages the first time you run the notebook
# %pip install -q pandas numpy tqdm sacrebleu rouge-score nltk bert-score sentence-transformers transformers torch

In [2]:
import ast
import glob
import json
import math
import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from rouge_score import rouge_scorer
from sacrebleu.metrics import CHRF, BLEU
from nltk.translate.meteor_score import meteor_score
import nltk

from bert_score import BERTScorer
from transformers import AutoModel, AutoTokenizer

nltk.download("wordnet")
nltk.download("omw-1.4")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)


[nltk_data] Downloading package wordnet to /home/vramon/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/vramon/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
# -----------------------
# Configuration
# -----------------------
CSV_GLOB = "./generations__*.csv"   # change if needed
OUTPUT_DIR = Path("./evaluation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda:1" if USE_CUDA else "cpu"

BERT_EMB_MODEL = "bert-base-multilingual-cased"
E5_MODEL = "intfloat/multilingual-e5-base"
BERTSCORE_MODEL = "bert-base-multilingual-cased"

# If you only want test:
FILTER_SPLITS = None  # e.g. ["test"]

print("CUDA available:", USE_CUDA)
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA L40


In [4]:
# -----------------------
# Parsing helpers
# -----------------------
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def parse_lexicalisations(value):
    """
    Expected CSV format:
    '[{"Id1": "..."}, {"Id2": "..."}, {"Id3": "..."}]'
    Returns:
        [{"lid": "Id1", "text": "..."}, ...]
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []

    if isinstance(value, list):
        raw = value
    else:
        s = str(value).strip()
        if not s:
            return []
        try:
            raw = json.loads(s)
        except Exception:
            raw = ast.literal_eval(s)

    refs = []
    if isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                for lid, text in item.items():
                    refs.append({"lid": str(lid), "text": normalize_text(text)})
            elif isinstance(item, str):
                refs.append({"lid": "", "text": normalize_text(item)})
    return [r for r in refs if r["text"]]

def choose_candidate_text(row):
    cand = normalize_text(row.get("extracted_verbalization", ""))
    if cand:
        return cand
    return normalize_text(row.get("raw_generation", ""))

def load_generation_csvs(csv_glob=CSV_GLOB):
    paths = sorted(glob.glob(csv_glob))
    if not paths:
        raise FileNotFoundError(f"No CSV files found for pattern: {csv_glob}")

    frames = []
    for path in paths:
        df = pd.read_csv(path)
        df["source_csv"] = Path(path).name
        if "model_name" not in df.columns:
            stem = Path(path).stem
            df["model_name"] = stem.replace("generations__", "").replace("__", "/")
        frames.append(df)

    data = pd.concat(frames, ignore_index=True)

    if FILTER_SPLITS is not None and "split" in data.columns:
        data = data[data["split"].isin(FILTER_SPLITS)].copy()

    data["candidate_text"] = data.apply(choose_candidate_text, axis=1)
    data["refs_struct"] = data["lexicalisations"].apply(parse_lexicalisations)
    data["references"] = data["refs_struct"].apply(lambda xs: [x["text"] for x in xs])
    data["reference_lids"] = data["refs_struct"].apply(lambda xs: [x["lid"] for x in xs])
    data["num_references"] = data["references"].apply(len)

    data = data[data["candidate_text"].str.len() > 0].copy()
    data = data[data["num_references"] > 0].copy()

    return data

df = load_generation_csvs()
print(df.shape)
display(df.head(3))


(113856, 33)


,lang,split,category,eid,size,num_triples,triple_bucket,xml_file,xml_path,align_key,num_lexicalisations,lexicalisations,triples,triples_struct,prompt,messages,model_name,raw_generation,extracted_verbalization,generation_status,generation_error,latency_sec,timestamp_utc,source_csv,base_model_name,base_model_family,experiment_name,adapter_dir,candidate_text,refs_struct,references,reference_lids,num_references
0,ca,test,Scientist,Id620,1,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO_BT/test/rdf-to-text-generation-test-data-with-refs-en.xml,test|||Scientist|||Id620|||1,3,"[{""Id1"": ""Indian és el nom demonímic de la gent de l'Índia.""}, {""Id2"": ""El nom demonim per a una persona de l'Índia és indi.""}, {""Id3"": ""Indià és el gentilici de la gent de l'Índia.""}]","[""Índia | Demònim | Indi""]","[{""subject"": ""Índia"", ""predicate"": ""Demònim"", ""object"": ""Indi"", ""raw"": ""Índia | Demònim | Indi""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...",HuggingFaceTB/SmolLM3-3B,"<think>\n\n</think>\nLa verbalització final és: [Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia ...","Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia parlamentària.",ok,NaN,1.5423,2026-03-19T15:03:17.950684+00:00,generations__HuggingFaceTB__SmolLM3-3B.csv,NaN,NaN,NaN,NaN,"Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia parlamentària.","[{'lid': 'Id1', 'text': 'Indian és el nom demonímic de la gent de l'Índia.'}, {'lid': 'Id2', 'text': 'El nom demonim per a una persona de l'Índia és indi.'}, {'lid': 'Id3', 'text': 'Indià és el ge...","[Indian és el nom demonímic de la gent de l'Índia., El nom demonim per a una persona de l'Índia és indi., Indià és el gentilici de la gent de l'Índia.]","[Id1, Id2, Id3]",3
1,ca,test,Scientist,Id798,1,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO_BT/test/rdf-to-text-generation-test-data-with-refs-en.xml,test|||Scientist|||Id798|||1,2,"[{""Id1"": ""Tokat es troba a Turquia.""}, {""Id2"": ""Tokat es troba al país de Turquia.""}]","[""Tokat | País | Turquia""]","[{""subject"": ""Tokat"", ""predicate"": ""País"", ""object"": ""Turquia"", ""raw"": ""Tokat | País | Turquia""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...",HuggingFaceTB/SmolLM3-3B,<think>\n\n</think>\nLa verbalització final és: [Tokat situada en Turquía.],Tokat situada en Turquía.,ok,NaN,0.6848,2026-03-19T15:03:18.635676+00:00,generations__HuggingFaceTB__SmolLM3-3B.csv,NaN,NaN,NaN,NaN,Tokat situada en Turquía.,"[{'lid': 'Id1', 'text': 'Tokat es troba a Turquia.'}, {'lid': 'Id2', 'text': 'Tokat es troba al país de Turquia.'}]","[Tokat es troba a Turquia., Tokat es troba al país de Turquia.]","[Id1, Id2]",2
2,ca,test,Scientist,Id121,1,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO_BT/test/rdf-to-text-generation-test-data-with-refs-en.xml,test|||Scientist|||Id121|||1,3,"[{""Id1"": ""Turk és el dimoni per als residents de Turquia.""}, {""Id2"": ""Un habitant de Turquia s'anomen

In [5]:
# -----------------------
# Metric setup
# -----------------------
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
chrf = CHRF(word_order=2)  # chrF++
bleu = BLEU(effective_order=True)

# BERTScore
bertscorer = BERTScorer(
    model_type=BERTSCORE_MODEL,
    lang=None,
    rescale_with_baseline=False,
    device=DEVICE,
)

# Embedding models
bert_tok = AutoTokenizer.from_pretrained(BERT_EMB_MODEL)
bert_model = AutoModel.from_pretrained(BERT_EMB_MODEL).to(DEVICE)
bert_model.eval()

e5_tok = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL).to(DEVICE)
e5_model.eval()

print("Loaded embedding models on:", DEVICE)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding models on: cuda:1


In [6]:
# -----------------------
# Embedding helpers
# -----------------------
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

@torch.inference_mode()
def encode_texts_mean(texts, tokenizer, model, prefix=None, batch_size=32):
    all_vecs = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        if prefix is not None:
            batch = [prefix + t for t in batch]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(DEVICE)

        out = model(**enc)
        vecs = mean_pool(out.last_hidden_state, enc["attention_mask"])
        vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs.detach().cpu())

    return torch.cat(all_vecs, dim=0)

bert_cache = {}
e5_cache = {}

def get_cached_embedding(text, kind="bert"):
    if kind == "bert":
        if text not in bert_cache:
            bert_cache[text] = encode_texts_mean([text], bert_tok, bert_model, prefix=None, batch_size=1)[0]
        return bert_cache[text]
    elif kind == "e5_query":
        key = ("query", text)
        if key not in e5_cache:
            e5_cache[key] = encode_texts_mean([text], e5_tok, e5_model, prefix="query: ", batch_size=1)[0]
        return e5_cache[key]
    elif kind == "e5_passage":
        key = ("passage", text)
        if key not in e5_cache:
            e5_cache[key] = encode_texts_mean([text], e5_tok, e5_model, prefix="passage: ", batch_size=1)[0]
        return e5_cache[key]
    else:
        raise ValueError(kind)

def cosine_from_cached(a, b):
    return float(torch.dot(a, b).item())


In [7]:
# -----------------------
# Per-instance metrics
# -----------------------
def safe_tokenize_for_meteor(text):
    return normalize_text(text).split()

def best_bleu(candidate, refs):
    # Uses all references for BLEU, not best-reference selection
    score = bleu.sentence_score(candidate, refs).score / 100.0
    return float(score)

def best_rougeL(candidate, refs):
    best = -1.0
    best_ref = None
    for ref in refs:
        score = rouge.score(candidate, ref)["rougeL"].fmeasure
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def best_meteor(candidate, refs):
    cand_tok = safe_tokenize_for_meteor(candidate)
    best = -1.0
    best_ref = None
    for ref in refs:
        ref_tok = safe_tokenize_for_meteor(ref)
        score = meteor_score([ref_tok], cand_tok)
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def best_chrfpp(candidate, refs):
    # sacrebleu sentence_score expects hypothesis first, list of refs second
    best = -1.0
    best_ref = None
    for ref in refs:
        score = chrf.sentence_score(candidate, [ref]).score / 100.0
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def best_bertscore(candidate, refs):
    cands = [candidate] * len(refs)
    P, R, F = bertscorer.score(cands, refs)
    idx = int(torch.argmax(F).item())
    return {
        "bertscore_precision": float(P[idx].item()),
        "bertscore_recall": float(R[idx].item()),
        "bertscore_f1": float(F[idx].item()),
        "best_ref_bertscore": refs[idx],
    }

def best_bert_cosine(candidate, refs):
    c = get_cached_embedding(candidate, kind="bert")
    best = -1.0
    best_ref = None
    for ref in refs:
        r = get_cached_embedding(ref, kind="bert")
        score = cosine_from_cached(c, r)
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def best_e5_cosine(candidate, refs):
    c = get_cached_embedding(candidate, kind="e5_query")
    best = -1.0
    best_ref = None
    for ref in refs:
        r = get_cached_embedding(ref, kind="e5_passage")
        score = cosine_from_cached(c, r)
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def expansion_ratio_closest_to_one(candidate, refs):
    cand_len = max(len(candidate), 1)
    best_ratio = None
    best_ref = None
    best_dist = None

    for ref in refs:
        ref_len = max(len(ref), 1)
        ratio = cand_len / ref_len
        dist = abs(ratio - 1.0)
        if best_dist is None or dist < best_dist:
            best_dist = dist
            best_ratio = ratio
            best_ref = ref

    return float(best_ratio), best_ref, float(best_dist)

def evaluate_one_row(row):
    candidate = row["candidate_text"]
    refs = row["references"]

    bleu_sentence = best_bleu(candidate, refs)
    rougeL, rouge_ref = best_rougeL(candidate, refs)
    meteor, meteor_ref = best_meteor(candidate, refs)
    chrfpp, chrf_ref = best_chrfpp(candidate, refs)
    bertscore = best_bertscore(candidate, refs)
    bert_cos, bert_ref = best_bert_cosine(candidate, refs)
    e5_cos, e5_ref = best_e5_cosine(candidate, refs)
    expansion_ratio, expansion_ref, expansion_dist = expansion_ratio_closest_to_one(candidate, refs)

    return {
        "bleu_sentence": bleu_sentence,
        "rougeL_f1": rougeL,
        "meteor": meteor,
        "chrfpp": chrfpp,
        "bertscore_precision": bertscore["bertscore_precision"],
        "bertscore_recall": bertscore["bertscore_recall"],
        "bertscore_f1": bertscore["bertscore_f1"],
        "bert_cosine": bert_cos,
        "e5_cosine": e5_cos,
        "expansion_ratio": expansion_ratio,
        "expansion_abs_distance_from_1": expansion_dist,
        "best_ref_rougeL": rouge_ref,
        "best_ref_meteor": meteor_ref,
        "best_ref_chrfpp": chrf_ref,
        "best_ref_bertscore": bertscore["best_ref_bertscore"],
        "best_ref_bert_cosine": bert_ref,
        "best_ref_e5_cosine": e5_ref,
        "best_ref_expansion": expansion_ref,
    }


In [8]:
# -----------------------
# Run evaluation
# -----------------------
records = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    out = evaluate_one_row(row)
    base = row.to_dict()
    base.update(out)
    records.append(base)

eval_df = pd.DataFrame(records)
print(eval_df.shape)
display(eval_df.head(3))


  0%|          | 0/113856 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



(113856, 51)


,lang,split,category,eid,size,num_triples,triple_bucket,xml_file,xml_path,align_key,num_lexicalisations,lexicalisations,triples,triples_struct,prompt,messages,model_name,raw_generation,extracted_verbalization,generation_status,generation_error,latency_sec,timestamp_utc,source_csv,base_model_name,base_model_family,experiment_name,adapter_dir,candidate_text,refs_struct,references,reference_lids,num_references,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,best_ref_rougeL,best_ref_meteor,best_ref_chrfpp,best_ref_bertscore,best_ref_bert_cosine,best_ref_e5_cosine,best_ref_expansion
0,ca,test,Scientist,Id620,1,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO_BT/test/rdf-to-text-generation-test-data-with-refs-en.xml,test|||Scientist|||Id620|||1,3,"[{""Id1"": ""Indian és el nom demonímic de la gent de l'Índia.""}, {""Id2"": ""El nom demonim per a una persona de l'Índia és indi.""}, {""Id3"": ""Indià és el gentilici de la gent de l'Índia.""}]","[""Índia | Demònim | Indi""]","[{""subject"": ""Índia"", ""predicate"": ""Demònim"", ""object"": ""Indi"", ""raw"": ""Índia | Demònim | Indi""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...",HuggingFaceTB/SmolLM3-3B,"<think>\n\n</think>\nLa verbalització final és: [Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia ...","Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia parlamentària.",ok,NaN,1.5423,2026-03-19T15:03:17.950684+00:00,generations__HuggingFaceTB__SmolLM3-3B.csv,NaN,NaN,NaN,NaN,"Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia parlamentària.","[{'lid': 'Id1', 'text': 'Indian és el nom demonímic de la gent de l'Índia.'}, {'lid': 'Id2', 'text': 'El nom demonim per a una persona de l'Índia és indi.'}, {'lid': 'Id3', 'text': 'Indià és el ge...","[Indian és el nom demonímic de la gent de l'Índia., El nom demonim per a una persona de l'Índia és indi., Indià és el gentilici de la gent de l'Índia.]","[Id1, Id2, Id3]",3,0.015784,0.139535,0.080000,0.192736,0.684124,0.742278,0.712015,0.682420,0.867955,3.096154,2.096154,Indià és el gentilici de la gent de l'Índia.,El nom demonim per a una persona de l'Índia és indi.,El nom demonim per a una persona de l'Índia és indi.,Indià és el gentilici de la gent de l'Índia.,Indian és el nom demonímic de la gent de l'Índia.,Indian és el nom demonímic de la gent de l'Índia.,El nom demonim per a una persona de l'Índia és indi.
1,ca,test,Scientist,Id798,1,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO_BT/test/rdf-to-text-generation-test-data-with-refs-en.xml,test|||Scientist|||Id798|||1,2,"[{""Id1"": ""Tokat es troba a Turquia.""}, {""Id2"": ""Tokat es troba al país de Turquia.""}]","[""Tokat | País | Turquia""]","[{""subject"": ""Tokat"", ""predicate"": ""País"", ""object"": ""Turquia"", ""raw"": ""Tokat | País | Turquia""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...",HuggingFaceTB/SmolLM3-3

In [9]:
# -----------------------
# Save instance-level results
# -----------------------
instance_cols_first = [
    "model_name", "lang", "split", "category", "eid", "align_key",
    "num_triples", "num_references", "candidate_text", "references",
    "bleu_sentence", "rougeL_f1", "meteor", "chrfpp",
    "bertscore_precision", "bertscore_recall", "bertscore_f1",
    "bert_cosine", "e5_cosine", "expansion_ratio", "expansion_abs_distance_from_1"
]
instance_cols = [c for c in instance_cols_first if c in eval_df.columns] + [c for c in eval_df.columns if c not in instance_cols_first]

eval_df.to_csv(OUTPUT_DIR / "instance_level_metrics.csv", index=False)
display(eval_df[instance_cols].head(10))


,model_name,lang,split,category,eid,align_key,num_triples,num_references,candidate_text,references,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,size,triple_bucket,xml_file,xml_path,num_lexicalisations,lexicalisations,triples,triples_struct,prompt,messages,raw_generation,extracted_verbalization,generation_status,generation_error,latency_sec,timestamp_utc,source_csv,base_model_name,base_model_family,experiment_name,adapter_dir,refs_struct,reference_lids,best_ref_rougeL,best_ref_meteor,best_ref_chrfpp,best_ref_bertscore,best_ref_bert_cosine,best_ref_e5_cosine,best_ref_expansion
0,HuggingFaceTB/SmolLM3-3B,ca,test,Scientist,Id620,test|||Scientist|||Id620|||1,1,3,"Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia parlamentària.","[Indian és el nom demonímic de la gent de l'Índia., El nom demonim per a una persona de l'Índia és indi., Indià és el gentilici de la gent de l'Índia.]",0.015784,0.139535,0.080000,0.192736,0.684124,0.742278,0.712015,0.682420,0.867955,3.096154,2.096154,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO_BT/test/rdf-to-text-generation-test-data-with-refs-en.xml,3,"[{""Id1"": ""Indian és el nom demonímic de la gent de l'Índia.""}, {""Id2"": ""El nom demonim per a una persona de l'Índia és indi.""}, {""Id3"": ""Indià és el gentilici de la gent de l'Índia.""}]","[""Índia | Demònim | Indi""]","[{""subject"": ""Índia"", ""predicate"": ""Demònim"", ""object"": ""Indi"", ""raw"": ""Índia | Demònim | Indi""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...","<think>\n\n</think>\nLa verbalització final és: [Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia ...","Índia, Demònim, Indi, va ser un país que ha tenut una gran influència cultural i política en l'Asia meridional, i que actualment és una democràcia parlamentària.",ok,NaN,1.5423,2026-03-19T15:03:17.950684+00:00,generations__HuggingFaceTB__SmolLM3-3B.csv,NaN,NaN,NaN,NaN,"[{'lid': 'Id1', 'text': 'Indian és el nom demonímic de la gent de l'Índia.'}, {'lid': 'Id2', 'text': 'El nom demonim per a una persona de l'Índia és indi.'}, {'lid': 'Id3', 'text': 'Indià és el ge...","[Id1, Id2, Id3]",Indià és el gentilici de la gent de l'Índia.,El nom demonim per a una persona de l'Índia és indi.,El nom demonim per a una persona de l'Índia és indi.,Indià és el gentilici de la gent de l'Índia.,Indian és el nom demonímic de la gent de l'Índia.,Indian és el nom demonímic de la gent de l'Índia.,El nom demonim per a una persona de l'Índia és indi.
1,HuggingFaceTB/SmolLM3-3B,ca,test,Scientist,Id798,test|||Scientist|||Id798|||1,1,2,Tokat situada en Turquía.,"[Tokat es troba a Turquia., Tokat es troba al país de Turquia.]",0.104006,0.400000,0.102041,0.286316,0.835274,0.816127,0.825590,0.672475,0.909622,1.000000,0.000000,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO_BT/test/rdf-to-text-generation-test-data-with-refs-en.xml,2,"[{""Id1"": ""Tokat es troba a Turquia.""}, {""Id2"": ""Tokat es troba al país de Turquia.""}]","[""Tokat | País | Turquia""]","[{""subject"": ""Tokat"", ""predicate"": ""País"", ""object"": ""Turquia"", ""raw"": ""Tokat | País | Turquia""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"

In [10]:
# -----------------------
# Corpus-BLEU helpers and summary by model × language
# -----------------------
def corpus_bleu_for_group(group_df):
    candidates = group_df["candidate_text"].fillna("").astype(str).tolist()
    refs_per_instance = group_df["references"].tolist()
    if not candidates or not refs_per_instance:
        return np.nan

    max_refs = max((len(r) for r in refs_per_instance), default=0)
    if max_refs == 0:
        return np.nan

    refs_by_index = []
    for ref_idx in range(max_refs):
        ref_stream = []
        for refs in refs_per_instance:
            if ref_idx < len(refs):
                ref_stream.append(refs[ref_idx])
            else:
                ref_stream.append(refs[-1] if refs else "")
        refs_by_index.append(ref_stream)

    return bleu.corpus_score(candidates, refs_by_index).score / 100.0

summary_ml = (
    eval_df
    .groupby(["model_name", "lang"], dropna=False)
    .agg(
        n=("candidate_text", "size"),
        bleu_sentence=("bleu_sentence", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        meteor=("meteor", "mean"),
        chrfpp=("chrfpp", "mean"),
        bertscore_precision=("bertscore_precision", "mean"),
        bertscore_recall=("bertscore_recall", "mean"),
        bertscore_f1=("bertscore_f1", "mean"),
        bert_cosine=("bert_cosine", "mean"),
        e5_cosine=("e5_cosine", "mean"),
        expansion_ratio=("expansion_ratio", "mean"),
        expansion_abs_distance_from_1=("expansion_abs_distance_from_1", "mean"),
    )
    .reset_index()
)

corpus_bleu_ml = (
    eval_df
    .groupby(["model_name", "lang"], dropna=False)
    .apply(corpus_bleu_for_group)
    .reset_index(name="bleu_corpus")
)

summary_ml = (
    summary_ml
    .merge(corpus_bleu_ml, on=["model_name", "lang"], how="left")
    .sort_values(["model_name", "lang"])
)

summary_ml.to_csv(OUTPUT_DIR / "summary_by_model_lang.csv", index=False)
summary_ml.to_excel(OUTPUT_DIR / "summary_by_model_lang.xlsx", index=False)
display(summary_ml)


/tmp/ipykernel_1893199/3339232330.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(corpus_bleu_for_group)


,model_name,lang,n,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,bleu_corpus
0,HuggingFaceTB/SmolLM3-3B,ca,1779,0.238594,0.480321,0.419672,0.521935,0.827188,0.840293,0.833145,0.856033,0.922447,1.151119,0.245710,0.238673
1,HuggingFaceTB/SmolLM3-3B,en,1779,0.451109,0.633027,0.632490,0.679062,0.884152,0.890657,0.887196,0.915652,0.917536,1.015022,0.080342,0.454573
2,HuggingFaceTB/SmolLM3-3B,en_bt,1779,0.445132,0.631732,0.632309,0.673655,0.882737,0.891767,0.887025,0.914097,0.916772,1.024784,0.086914,0.446470
3,HuggingFaceTB/SmolLM3-3B,es,1779,0.334540,0.543489,0.486087,0.586326,0.853640,0.865805,0.859274,0.886883,0.921673,1.073729,0.157044,0.364053
4,HuggingFaceTB__SmolLM3-3B__CA-silver-filtered__CA-silver-filtered,ca,1779,0.307119,0.458863,0.500715,0.553083,0.809149,0.836230,0.821530,0.837551,0.924456,1.556596,0.608172,0.257409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,Qwen__Qwen3-4B-Instruct-2507__ES-silver-filtered__ES-silver-filtered,es,1779,0.498162,0.639941,0.626490,0.675386,0.897319,0.895206,0.896104,0.921069,0.930218,0.984407,0.069019,0.515377
60,Qwen__Qwen3-4B-Instruct-2507__ES-silver__ES-silver,ca,1779,0.358173,0.551189,0.506983,0.590425,0.869721,0.866394,0.867836,0.891044,0.931162,0.978072,0.085758,0.365437
61,Qwen__Qwen3-4B-Instruct-2507__ES-silver__ES-silver,en,1779,0.522013,0.678746,0.656093,0.701807,0.906184,0.901824,0.903840,0.926206,0.917295,0.968675,0.068719,0.526149
62,Qwen__Qwen3-4B-Instruct-2507__ES-silver__ES-silver,en_bt,1779,0.527006,0.685284,0.667416,0.707940,0.907163,0.905209,0.906027,0.928141,0.917174,0.972764,0.067919,0.535197


In [11]:
# -----------------------
# Optional: summary by model × language × split
# -----------------------
summary_mls = (
    eval_df
    .groupby(["model_name", "lang", "split"], dropna=False)
    .agg(
        n=("candidate_text", "size"),
        bleu_sentence=("bleu_sentence", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        meteor=("meteor", "mean"),
        chrfpp=("chrfpp", "mean"),
        bertscore_precision=("bertscore_precision", "mean"),
        bertscore_recall=("bertscore_recall", "mean"),
        bertscore_f1=("bertscore_f1", "mean"),
        bert_cosine=("bert_cosine", "mean"),
        e5_cosine=("e5_cosine", "mean"),
        expansion_ratio=("expansion_ratio", "mean"),
        expansion_abs_distance_from_1=("expansion_abs_distance_from_1", "mean"),
    )
    .reset_index()
)

corpus_bleu_mls = (
    eval_df
    .groupby(["model_name", "lang", "split"], dropna=False)
    .apply(corpus_bleu_for_group)
    .reset_index(name="bleu_corpus")
)

summary_mls = (
    summary_mls
    .merge(corpus_bleu_mls, on=["model_name", "lang", "split"], how="left")
    .sort_values(["model_name", "lang", "split"])
)

summary_mls.to_csv(OUTPUT_DIR / "summary_by_model_lang_split.csv", index=False)
display(summary_mls)


/tmp/ipykernel_1893199/4017172665.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(corpus_bleu_for_group)


,model_name,lang,split,n,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,bleu_corpus
0,HuggingFaceTB/SmolLM3-3B,ca,test,1779,0.238594,0.480321,0.419672,0.521935,0.827188,0.840293,0.833145,0.856033,0.922447,1.151119,0.245710,0.238673
1,HuggingFaceTB/SmolLM3-3B,en,test,1779,0.451109,0.633027,0.632490,0.679062,0.884152,0.890657,0.887196,0.915652,0.917536,1.015022,0.080342,0.454573
2,HuggingFaceTB/SmolLM3-3B,en_bt,test,1779,0.445132,0.631732,0.632309,0.673655,0.882737,0.891767,0.887025,0.914097,0.916772,1.024784,0.086914,0.446470
3,HuggingFaceTB/SmolLM3-3B,es,test,1779,0.334540,0.543489,0.486087,0.586326,0.853640,0.865805,0.859274,0.886883,0.921673,1.073729,0.157044,0.364053
4,HuggingFaceTB__SmolLM3-3B__CA-silver-filtered__CA-silver-filtered,ca,test,1779,0.307119,0.458863,0.500715,0.553083,0.809149,0.836230,0.821530,0.837551,0.924456,1.556596,0.608172,0.257409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,Qwen__Qwen3-4B-Instruct-2507__ES-silver-filtered__ES-silver-filtered,es,test,1779,0.498162,0.639941,0.626490,0.675386,0.897319,0.895206,0.896104,0.921069,0.930218,0.984407,0.069019,0.515377
60,Qwen__Qwen3-4B-Instruct-2507__ES-silver__ES-silver,ca,test,1779,0.358173,0.551189,0.506983,0.590425,0.869721,0.866394,0.867836,0.891044,0.931162,0.978072,0.085758,0.365437
61,Qwen__Qwen3-4B-Instruct-2507__ES-silver__ES-silver,en,test,1779,0.522013,0.678746,0.656093,0.701807,0.906184,0.901824,0.903840,0.926206,0.917295,0.968675,0.068719,0.526149
62,Qwen__Qwen3-4B-Instruct-2507__ES-silver__ES-silver,en_bt,test,1779,0.527006,0.685284,0.667416,0.707940,0.907163,0.905209,0.906027,0.928141,0.917174,0.972764,0.067919,0.535197


In [12]:
# -----------------------
# Optional: summary by model × language × category
# -----------------------
summary_mlc = (
    eval_df
    .groupby(["model_name", "lang", "category"], dropna=False)
    .agg(
        n=("candidate_text", "size"),
        bleu_sentence=("bleu_sentence", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        meteor=("meteor", "mean"),
        chrfpp=("chrfpp", "mean"),
        bertscore_precision=("bertscore_precision", "mean"),
        bertscore_recall=("bertscore_recall", "mean"),
        bertscore_f1=("bertscore_f1", "mean"),
        bert_cosine=("bert_cosine", "mean"),
        e5_cosine=("e5_cosine", "mean"),
        expansion_ratio=("expansion_ratio", "mean"),
        expansion_abs_distance_from_1=("expansion_abs_distance_from_1", "mean"),
    )
    .reset_index()
)

corpus_bleu_mlc = (
    eval_df
    .groupby(["model_name", "lang", "category"], dropna=False)
    .apply(corpus_bleu_for_group)
    .reset_index(name="bleu_corpus")
)

summary_mlc = (
    summary_mlc
    .merge(corpus_bleu_mlc, on=["model_name", "lang", "category"], how="left")
    .sort_values(["model_name", "lang", "category"])
)

summary_mlc.to_csv(OUTPUT_DIR / "summary_by_model_lang_category.csv", index=False)
display(summary_mlc.head(20))


/tmp/ipykernel_1893199/3329380674.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(corpus_bleu_for_group)


,model_name,lang,category,n,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,bleu_corpus
0,HuggingFaceTB/SmolLM3-3B,ca,Airport,95,0.192453,0.459498,0.394372,0.515207,0.840334,0.845630,0.842648,0.872449,0.925319,1.123831,0.186247,0.201684
1,HuggingFaceTB/SmolLM3-3B,ca,Artist,109,0.255511,0.557925,0.438327,0.559041,0.856826,0.866181,0.861195,0.889439,0.923133,1.035345,0.123480,0.262001
2,HuggingFaceTB/SmolLM3-3B,ca,Astronaut,82,0.258812,0.541881,0.471654,0.540150,0.846863,0.856670,0.851407,0.891805,0.920302,1.009715,0.108046,0.252802
3,HuggingFaceTB/SmolLM3-3B,ca,Athlete,50,0.215578,0.504936,0.400832,0.529917,0.833072,0.850838,0.841532,0.870667,0.931326,1.300627,0.365692,0.176797
4,HuggingFaceTB/SmolLM3-3B,ca,Building,46,0.362963,0.450872,0.447934,0.522642,0.828272,0.836499,0.832055,0.889111,0.929748,0.995863,0.072469,0.382982
5,HuggingFaceTB/SmolLM3-3B,ca,CelestialBody,49,0.064436,0.349629,0.103773,0.264121,0.730633,0.686059,0.705622,0.659250,0.881380,0.741033,0.662813,0.062099
6,HuggingFaceTB/SmolLM3-3B,ca,City,83,0.248913,0.503707,0.470154,0.537421,0.832195,0.837118,0.834400,0.859229,0.928172,1.089991,0.135721,0.215462
7,HuggingFaceTB/SmolLM3-3B,ca,ComicsCharacter,30,0.192384,0.382305,0.311349,0.415196,0.824430,0.823073,0.823624,0.839159,0.902574,0.962307,0.087096,0.199162
8,HuggingFaceTB/SmolLM3-3B,ca,Company,66,0.199132,0.444055,0.332230,0.474028,0.832123,0.846889,0.838743,0.850414,0.926305,1.065985,0.173278,0.183508
9,HuggingFaceTB/SmolLM3-3B,ca,Film,264,0.226162,0.467322,0.400048,0.510764,0.795927,0.820482,0.806974,0.806859,0.915933,1.443758,0.499088,0.182178


## Notes

- `candidate_text` uses `extracted_verbalization` when available; otherwise it falls back to `raw_generation`.
- `expansion_ratio` is computed at character level:

  \[
  \text{expansion ratio} = \frac{|candidate|}{|reference|}
  \]

  and the selected reference is the one minimizing:

  \[
  |\text{expansion ratio} - 1|
  \]

- If you want token-level expansion ratio instead, replace `len(candidate)` and `len(ref)` with token counts.
- `bert_cosine` and `e5_cosine` are cosine similarities over normalized embeddings, so they are in `[-1, 1]`, though in practice they should be much higher for valid verbalisations.


In [16]:
# check_backtranslation_diff.py
# Checks that back-translated English is present and is not trivially identical
# to the original English already in the XML.
#
# Assumptions:
# - Original English verbalisations are stored in <lex lang="en" ...>
# - Back-translated English verbalisations are stored in <lex lang="en_bt" ...>
# - Original English triples are stored in <originaltripleset> or <modifiedtripleset>
# - Back-translated English triples are stored in <modifiedtripleset>
# - Catalan triples are stored in <catalantripleset>
#
# If your setup is different, adjust ORIGINAL_EN_LEX_LANG / BT_EN_LEX_LANG
# and the node names in the helper functions below.

import os
import re
import csv
import xml.etree.ElementTree as ET
from collections import Counter
from difflib import SequenceMatcher
from typing import Dict, List, Optional, Tuple

from tqdm.auto import tqdm


# ============================================================
# CONFIG
# ============================================================
DATASET_ROOT = "../../WebNLG_CO_BT"
OUT_CSV = "./backtranslation_sanity_check.csv"

# For verbalisations:
# Change these if needed.
ORIGINAL_EN_LEX_LANG = "en"
BT_EN_LEX_LANG = "en_bt"   # <- set this to whatever lang/code you used for backtranslated EN

# For triples:
ORIGINAL_EN_TRIPLESET_TAG_CANDIDATES = ["originaltripleset", "modifiedtripleset"]
BT_EN_TRIPLESET_TAG = "enbttripleset"
CA_TRIPLESET_TAG = "catalantripleset"

TRIPLE_SPLIT = " | "


# ============================================================
# TEXT NORMALIZATION / SIMPLE SIMILARITY
# ============================================================
def norm_text(s: str) -> str:
    s = (s or "").strip()
    s = s.replace("_", " ")
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def strip_punct_spaces(s: str) -> str:
    s = norm_text(s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def sim_ratio(a: str, b: str) -> float:
    return SequenceMatcher(None, norm_text(a), norm_text(b)).ratio()

def token_jaccard(a: str, b: str) -> float:
    ta = set(strip_punct_spaces(a).split())
    tb = set(strip_punct_spaces(b).split())
    if not ta and not tb:
        return 1.0
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

def classify_text_pair(a: str, b: str) -> str:
    """
    Heuristic labels:
    - EXACT: exactly the same after normalization
    - NEAR_EXACT: very similar, suspicious
    - DIFFERENT: clearly different
    """
    na = norm_text(a)
    nb = norm_text(b)

    if na == nb:
        return "EXACT"

    sp_a = strip_punct_spaces(a)
    sp_b = strip_punct_spaces(b)
    if sp_a == sp_b:
        return "EXACT_MINUS_PUNCT"

    sr = sim_ratio(a, b)
    jac = token_jaccard(a, b)

    if sr >= 0.97 or jac >= 0.97:
        return "NEAR_EXACT"

    return "DIFFERENT"


# ============================================================
# XML HELPERS
# ============================================================
def iter_xml_files(root_dir: str):
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if fn.lower().endswith(".xml"):
                yield os.path.join(dirpath, fn)

def parse_triple_line(line: str) -> Tuple[str, str, str]:
    parts = [p.strip() for p in line.split(TRIPLE_SPLIT, 2)]
    if len(parts) != 3:
        parts = [p.strip() for p in line.split("|", 2)]
    if len(parts) != 3:
        raise ValueError(f"Bad triple: {line!r}")
    return parts[0], parts[1], parts[2]

def find_lexes(entry_el: ET.Element, lang_code: str) -> List[Tuple[str, str]]:
    out = []
    for lex in entry_el.findall("lex"):
        if lex.get("lang") == lang_code:
            lid = lex.get("lid") or "Id1"
            txt = (lex.text or "").strip()
            out.append((lid, txt))
    return out

def find_lex_by_lid(entry_el: ET.Element, lang_code: str, lid: str) -> Optional[str]:
    for lex in entry_el.findall("lex"):
        if lex.get("lang") == lang_code and (lex.get("lid") or "Id1") == lid:
            return (lex.text or "").strip()
    return None

def find_first_existing_node(entry_el: ET.Element, tags: List[str]) -> Optional[ET.Element]:
    for tag in tags:
        node = entry_el.find(tag)
        if node is not None:
            return node
    return None

def get_triples(node: Optional[ET.Element]) -> List[str]:
    if node is None:
        return []
    out = []
    for child in list(node):
        txt = (child.text or "").strip()
        if txt:
            out.append(txt)
    return out


# ============================================================
# TRIPLE COMPARISON
# ============================================================
def normalize_triple_token(token: str) -> str:
    t = norm_text(token)
    t = re.sub(r"\s+", " ", t)
    return t

def normalize_triple_line(line: str) -> str:
    try:
        s, p, o = parse_triple_line(line)
        s = normalize_triple_token(s)
        p = normalize_triple_token(p)
        o = normalize_triple_token(o)
        return f"{s} | {p} | {o}"
    except Exception:
        return norm_text(line)

def compare_triple_lists(orig_triples: List[str], bt_triples: List[str]) -> Dict[str, object]:
    orig_norm = [normalize_triple_line(x) for x in orig_triples]
    bt_norm = [normalize_triple_line(x) for x in bt_triples]

    same_count = len(orig_norm) == len(bt_norm)
    exact_list_match = orig_norm == bt_norm
    exact_set_match = set(orig_norm) == set(bt_norm)

    joined_orig = " || ".join(orig_norm)
    joined_bt = " || ".join(bt_norm)

    return {
        "orig_count": len(orig_norm),
        "bt_count": len(bt_norm),
        "same_count": int(same_count),
        "exact_list_match": int(exact_list_match),
        "exact_set_match": int(exact_set_match),
        "sim_ratio": round(sim_ratio(joined_orig, joined_bt), 4),
        "token_jaccard": round(token_jaccard(joined_orig, joined_bt), 4),
        "classification": classify_text_pair(joined_orig, joined_bt),
    }


# ============================================================
# MAIN CHECK
# ============================================================
ROWS = []
summary = Counter()

xml_files = sorted(iter_xml_files(DATASET_ROOT))

for xml_path in tqdm(xml_files, desc="Checking XMLs", unit="file"):
    try:
        tree = ET.parse(xml_path)
    except Exception as e:
        ROWS.append({
            "xml_path": xml_path,
            "category": "",
            "eid": "",
            "item_type": "file",
            "lid": "",
            "status": "FILE_PARSE_ERROR",
            "detail": repr(e),
        })
        summary["FILE_PARSE_ERROR"] += 1
        continue

    root = tree.getroot()
    entries_parent = root.find("entries")
    if entries_parent is None:
        ROWS.append({
            "xml_path": xml_path,
            "category": "",
            "eid": "",
            "item_type": "file",
            "lid": "",
            "status": "NO_ENTRIES_NODE",
            "detail": "",
        })
        summary["NO_ENTRIES_NODE"] += 1
        continue

    for entry in entries_parent.findall("entry"):
        category = entry.get("category", "")
        eid = entry.get("eid", "")

        # ----------------------------------------------------
        # 1) VERBALISATIONS
        # ----------------------------------------------------
        orig_lexes = find_lexes(entry, ORIGINAL_EN_LEX_LANG)
        bt_lexes = find_lexes(entry, BT_EN_LEX_LANG)

        bt_lids = {lid for lid, _ in bt_lexes}

        if not bt_lexes:
            ROWS.append({
                "xml_path": xml_path,
                "category": category,
                "eid": eid,
                "item_type": "lex",
                "lid": "",
                "status": "NO_BT_EN_LEX",
                "detail": f"No <lex lang='{BT_EN_LEX_LANG}'> found",
            })
            summary["NO_BT_EN_LEX"] += 1
        else:
            for lid, bt_text in bt_lexes:
                orig_text = find_lex_by_lid(entry, ORIGINAL_EN_LEX_LANG, lid)

                if orig_text is None:
                    status = "BT_LID_WITHOUT_ORIG_EN"
                    summary[status] += 1
                    ROWS.append({
                        "xml_path": xml_path,
                        "category": category,
                        "eid": eid,
                        "item_type": "lex",
                        "lid": lid,
                        "status": status,
                        "detail": bt_text,
                    })
                    continue

                cls = classify_text_pair(orig_text, bt_text)
                status = f"LEX_{cls}"
                summary[status] += 1

                ROWS.append({
                    "xml_path": xml_path,
                    "category": category,
                    "eid": eid,
                    "item_type": "lex",
                    "lid": lid,
                    "status": status,
                    "orig_text": orig_text,
                    "bt_text": bt_text,
                    "sim_ratio": round(sim_ratio(orig_text, bt_text), 4),
                    "token_jaccard": round(token_jaccard(orig_text, bt_text), 4),
                })

        # check missing BT lids for originals
        for lid, orig_text in orig_lexes:
            if lid not in bt_lids:
                ROWS.append({
                    "xml_path": xml_path,
                    "category": category,
                    "eid": eid,
                    "item_type": "lex",
                    "lid": lid,
                    "status": "ORIG_EN_WITHOUT_BT",
                    "detail": orig_text,
                })
                summary["ORIG_EN_WITHOUT_BT"] += 1

        # ----------------------------------------------------
        # 2) TRIPLES
        # ----------------------------------------------------
        orig_triple_node = find_first_existing_node(entry, ORIGINAL_EN_TRIPLESET_TAG_CANDIDATES)
        bt_triple_node = entry.find(BT_EN_TRIPLESET_TAG)
        ca_triple_node = entry.find(CA_TRIPLESET_TAG)

        orig_triples = get_triples(orig_triple_node)
        bt_triples = get_triples(bt_triple_node)
        ca_triples = get_triples(ca_triple_node)

        if not ca_triples:
            ROWS.append({
                "xml_path": xml_path,
                "category": category,
                "eid": eid,
                "item_type": "triples",
                "lid": "",
                "status": "NO_CA_TRIPLES",
                "detail": "",
            })
            summary["NO_CA_TRIPLES"] += 1

        if not bt_triples:
            ROWS.append({
                "xml_path": xml_path,
                "category": category,
                "eid": eid,
                "item_type": "triples",
                "lid": "",
                "status": "NO_BT_EN_TRIPLES",
                "detail": f"No <{BT_EN_TRIPLESET_TAG}> found or empty",
            })
            summary["NO_BT_EN_TRIPLES"] += 1
        elif not orig_triples:
            ROWS.append({
                "xml_path": xml_path,
                "category": category,
                "eid": eid,
                "item_type": "triples",
                "lid": "",
                "status": "NO_ORIG_EN_TRIPLES",
                "detail": "",
            })
            summary["NO_ORIG_EN_TRIPLES"] += 1
        else:
            cmp = compare_triple_lists(orig_triples, bt_triples)

            if cmp["exact_list_match"]:
                status = "TRIPLES_EXACT_LIST"
            elif cmp["exact_set_match"]:
                status = "TRIPLES_EXACT_SET"
            elif cmp["classification"] in {"EXACT", "EXACT_MINUS_PUNCT", "NEAR_EXACT"}:
                status = "TRIPLES_NEAR_EXACT"
            else:
                status = "TRIPLES_DIFFERENT"

            summary[status] += 1

            ROWS.append({
                "xml_path": xml_path,
                "category": category,
                "eid": eid,
                "item_type": "triples",
                "lid": "",
                "status": status,
                "orig_triple_count": cmp["orig_count"],
                "bt_triple_count": cmp["bt_count"],
                "same_count": cmp["same_count"],
                "exact_list_match": cmp["exact_list_match"],
                "exact_set_match": cmp["exact_set_match"],
                "sim_ratio": cmp["sim_ratio"],
                "token_jaccard": cmp["token_jaccard"],
                "orig_triples_joined": " || ".join(orig_triples),
                "bt_triples_joined": " || ".join(bt_triples),
            })



print("SUMMARY")
for k, v in sorted(summary.items()):
    print(f"{k}: {v}")

print("\nInterpretation:")
print("- LEX_EXACT / LEX_EXACT_MINUS_PUNCT / LEX_NEAR_EXACT are suspicious for back-translation.")
print("- TRIPLES_EXACT_LIST / TRIPLES_EXACT_SET / TRIPLES_NEAR_EXACT are suspicious for back-translation.")
print("- LEX_DIFFERENT and TRIPLES_DIFFERENT are the expected outcome if back-translation was actually performed.")

Checking XMLs:   0%|          | 0/185 [00:00<?, ?file/s]

SUMMARY
LEX_DIFFERENT: 30737
LEX_EXACT: 9591
LEX_EXACT_MINUS_PUNCT: 1554
LEX_NEAR_EXACT: 8460
NO_BT_EN_LEX: 114
ORIG_EN_WITHOUT_BT: 286
TRIPLES_DIFFERENT: 9398
TRIPLES_EXACT_LIST: 6173
TRIPLES_EXACT_SET: 66
TRIPLES_NEAR_EXACT: 2977

Interpretation:
- LEX_EXACT / LEX_EXACT_MINUS_PUNCT / LEX_NEAR_EXACT are suspicious for back-translation.
- TRIPLES_EXACT_LIST / TRIPLES_EXACT_SET / TRIPLES_NEAR_EXACT are suspicious for back-translation.
- LEX_DIFFERENT and TRIPLES_DIFFERENT are the expected outcome if back-translation was actually performed.
